# 📊 DỰ ÁN 1 — XÂY DỰNG MÔ HÌNH PHÂN LOẠI VÀ DỰ BÁO RỦI RO KHÁCH HÀNG VAY VỐN

**Notebook 02/07 — PostgreSQL Pipeline (Pipeline dữ liệu trên PostgreSQL)**

---

**🎯 Mục tiêu:** Xây dựng pipeline dữ liệu trên **PostgreSQL**: tạo bảng, import dữ liệu CSV, tạo view và bảng tổng hợp (aggregation) phục vụ làm sạch và phân tích ở các bước sau.

**📥 Input:** `data/raw/*.csv`; các script `sql/01_create_tables.sql` → `sql/05_indexes.sql`

**📤 Output:** CSDL PostgreSQL chứa các bảng gốc + các view/bảng tổng hợp.

**🔗 Pipeline:** 01. Data Understanding → **02. PostgreSQL Pipeline** → 03. Data Cleaning

## ⚙️ Chuẩn bị trước khi chạy (BẮT BUỘC cho mỗi thành viên)

Sau khi `git pull`, notebook này **chưa chạy được ngay**. Các thứ bên dưới KHÔNG nằm trong git (đã bị `.gitignore`) nên mỗi máy phải tự chuẩn bị **1 lần**:

**1. Cài thư viện Python:**
```
pip install -r requirements.txt
```

**2. Cài & bật PostgreSQL, rồi tạo database rỗng** (chỉ tạo *database*; các *bảng* để notebook tự tạo):
```
psql -U postgres -c "CREATE DATABASE credit_risk_db;"
```
> Hoặc trong pgAdmin: chuột phải **Databases → Create → Database** → đặt tên `credit_risk_db`.

**3. Tạo file `.env` ở thư mục gốc dự án** (copy từ mẫu `.env.example` rồi sửa mật khẩu):
```
cp .env.example .env
```
> Mở `.env`, đổi `DB_PASSWORD=your_password_here` thành **mật khẩu PostgreSQL của máy bạn**. Các thông số còn lại (`DB_NAME=credit_risk_db`...) giữ nguyên nếu bạn cài mặc định.

**4. Tải bộ dữ liệu Home Credit Default Risk (Kaggle) và bỏ 8 file CSV vào `data/raw/`:**
`application_train.csv`, `application_test.csv`, `bureau.csv`, `bureau_balance.csv`, `previous_application.csv`, `POS_CASH_balance.csv` (viết HOA `POS_CASH`), `installments_payments.csv`, `credit_card_balance.csv`.

Xong 4 bước → **Kernel → Restart & Run All**. Notebook chạy lại nhiều lần vẫn ra đúng số dòng (hàm import đã `TRUNCATE` bảng trước khi nạp).

---

## 1. Chuẩn bị môi trường & Kết nối Database

Chúng ta sẽ khai báo các thư viện cần thiết, load thông tin cấu hình kết nối database từ file `.env` cục bộ.

In [1]:
import os
import time
import psycopg2
from psycopg2 import sql
import pandas as pd
from dotenv import load_dotenv, find_dotenv

# Load file .env cấu hình database kết nối
# Tự động tìm kiếm file .env ở thư mục gốc bất kể thư mục chạy của Jupyter.
# override=True để luôn nạp lại giá trị mới nhất từ .env, tránh dùng nhầm
# biến môi trường cũ còn sót trong kernel (nguyên nhân lỗi auth khi kernel chưa restart).
load_dotenv(find_dotenv(), override=True)

# Lấy thông số kết nối từ biến môi trường
db_host = os.getenv('DB_HOST', 'localhost')
db_port = os.getenv('DB_PORT', '5432')
db_name = os.getenv('DB_NAME', 'credit_risk_db')
db_user = os.getenv('DB_USER', 'postgres')
db_password = os.getenv('DB_PASSWORD', 'postgres')

print("Thông số kết nối database:")
print(f"- Host: {db_host}")
print(f"- Port: {db_port}")
print(f"- Database: {db_name}")
print(f"- User: {db_user}")

Thông số kết nối database:
- Host: localhost
- Port: 5432
- Database: credit_risk_db
- User: postgres


In [2]:
def execute_sql_file(file_path, conn):
    """Hàm đọc và thực thi toàn bộ câu lệnh trong một file SQL"""
    start_time = time.time()
    with open(file_path, 'r', encoding='utf-8') as f:
        sql_script = f.read()
    
    try:
        with conn.cursor() as cursor:
            cursor.execute(sql_script)
        conn.commit()
        elapsed = time.time() - start_time
        print(f"✅ Thực thi thành công file: {file_path} ({elapsed:.2f} giây).")
    except Exception as e:
        conn.rollback()
        print(f"❌ Lỗi khi thực thi file {file_path}: {e}")
        raise e

## 2. Tạo cấu trúc bảng dữ liệu thô (Schema)

Chúng ta sẽ kết nối đến PostgreSQL và thực thi file `01_create_tables.sql` để xóa các bảng cũ (nếu có) và tạo mới cấu trúc 8 bảng dữ liệu thô.

In [3]:
# Khởi tạo kết nối tới database
conn = psycopg2.connect(
    host=db_host,
    port=db_port,
    database=db_name,
    user=db_user,
    password=db_password
)

# Chạy file tạo bảng thô
execute_sql_file('../sql/01_create_tables.sql', conn)

✅ Thực thi thành công file: ../sql/01_create_tables.sql (0.48 giây).


## 3. Pipeline Import dữ liệu tự động (Client-side COPY)

Để tránh lỗi phân quyền trên Windows khi PostgreSQL server đọc trực tiếp đường dẫn file, chúng ta sử dụng phương thức `copy_expert` của thư viện `psycopg2`. Phương thức này sẽ stream dữ liệu từ file CSV ở phía client-side vào PostgreSQL server.

> **Chạy lại an toàn (idempotent):** hàm import `TRUNCATE` bảng trước mỗi lần `COPY`, nên chạy lại notebook nhiều lần cũng không bị nạp trùng dữ liệu. `TRUNCATE` và `COPY` nằm chung một transaction — nếu COPY lỗi thì rollback cả hai, bảng giữ nguyên dữ liệu cũ.

In [4]:
def import_csv_to_db(table_name, csv_path, conn):
    """Hàm import file CSV vào bảng tương ứng sử dụng copy_expert.

    TRUNCATE bảng ngay trước khi COPY để hàm chạy lại được nhiều lần mà
    KHÔNG bị nạp trùng dữ liệu (idempotent). TRUNCATE và COPY nằm chung
    một transaction: nếu COPY lỗi thì rollback cả hai -> bảng giữ nguyên
    dữ liệu cũ, không rơi vào trạng thái rỗng dở dang.
    """
    start_time = time.time()
    print(f"Đang import dữ liệu vào bảng '{table_name}' từ file '{csv_path}'...")
    
    # Lệnh COPY FROM STDIN đọc luồng dữ liệu từ Client
    copy_query = f"COPY {table_name} FROM STDIN WITH (FORMAT CSV, HEADER, DELIMITER ',')"
    
    try:
        with conn.cursor() as cursor:
            # Xóa sạch dữ liệu cũ trước khi nạp -> tránh cộng dồn khi chạy lại.
            # Dùng sql.Identifier để đóng ngoặc kép tên bảng an toàn.
            cursor.execute(
                sql.SQL("TRUNCATE TABLE {} RESTART IDENTITY").format(
                    sql.Identifier(table_name)
                )
            )
            with open(csv_path, 'r', encoding='utf-8') as f:
                cursor.copy_expert(copy_query, f)
        conn.commit()
        elapsed_time = time.time() - start_time
        print(f"✅ Import thành công bảng '{table_name}' trong {elapsed_time:.2f} giây.\n")
    except Exception as e:
        conn.rollback()
        print(f"❌ Lỗi khi import bảng '{table_name}': {e}\n")
        raise e

In [5]:
# Danh sách mapping giữa tên bảng trong DB và đường dẫn file CSV thô
csv_mapping = {
    'application_train': '../data/raw/application_train.csv',
    'application_test': '../data/raw/application_test.csv',
    'bureau': '../data/raw/bureau.csv',
    'bureau_balance': '../data/raw/bureau_balance.csv',
    'previous_application': '../data/raw/previous_application.csv',
    'pos_cash_balance': '../data/raw/POS_CASH_balance.csv',
    'installments_payments': '../data/raw/installments_payments.csv',
    'credit_card_balance': '../data/raw/credit_card_balance.csv'
}

# Thực hiện import tự động
total_start = time.time()
for table, path in csv_mapping.items():
    if os.path.exists(path):
        import_csv_to_db(table, path, conn)
    else:
        print(f"⚠️ Không tìm thấy file CSV thô tại: {path}\n")

print(f"🎉 Hoàn thành import toàn bộ 8 bảng dữ liệu thô trong {time.time() - total_start:.2f} giây!")

Đang import dữ liệu vào bảng 'application_train' từ file '../data/raw/application_train.csv'...
✅ Import thành công bảng 'application_train' trong 9.18 giây.

Đang import dữ liệu vào bảng 'application_test' từ file '../data/raw/application_test.csv'...
✅ Import thành công bảng 'application_test' trong 2.05 giây.

Đang import dữ liệu vào bảng 'bureau' từ file '../data/raw/bureau.csv'...
✅ Import thành công bảng 'bureau' trong 19.79 giây.

Đang import dữ liệu vào bảng 'bureau_balance' từ file '../data/raw/bureau_balance.csv'...
✅ Import thành công bảng 'bureau_balance' trong 165.17 giây.

Đang import dữ liệu vào bảng 'previous_application' từ file '../data/raw/previous_application.csv'...
✅ Import thành công bảng 'previous_application' trong 34.82 giây.

Đang import dữ liệu vào bảng 'pos_cash_balance' từ file '../data/raw/POS_CASH_balance.csv'...
✅ Import thành công bảng 'pos_cash_balance' trong 29.66 giây.

Đang import dữ liệu vào bảng 'installments_payments' từ file '../data/raw/instal

## 4. Xây dựng Views, Bảng tổng hợp (Aggregation) & Chỉ mục (Indexes)

Sau khi đã có dữ liệu thô, chúng ta tiến hành chạy các script tạo View phân tích, bảng tổng hợp Materialized View phục vụ Feature Engineering, và thiết lập các Index để tăng tốc truy vấn.

In [6]:
print("1. Đang tạo các View phân tích...")
execute_sql_file('../sql/03_views.sql', conn)

print("2. Đang tạo các bảng tổng hợp (Materialized Views)...(Việc này có thể mất vài chục giây do phải tính toán trên lượng dữ liệu lớn)")
execute_sql_file('../sql/04_aggregation.sql', conn)

print("3. Đang tạo các chỉ mục tối ưu hóa hiệu năng JOIN (Indexes)...")
execute_sql_file('../sql/05_indexes.sql', conn)

1. Đang tạo các View phân tích...
✅ Thực thi thành công file: ../sql/03_views.sql (0.15 giây).
2. Đang tạo các bảng tổng hợp (Materialized Views)...(Việc này có thể mất vài chục giây do phải tính toán trên lượng dữ liệu lớn)
✅ Thực thi thành công file: ../sql/04_aggregation.sql (95.48 giây).
3. Đang tạo các chỉ mục tối ưu hóa hiệu năng JOIN (Indexes)...
✅ Thực thi thành công file: ../sql/05_indexes.sql (89.39 giây).


## 5. Kiểm tra và Đối chiếu số lượng dòng (Data Validation)

Chúng ta sẽ truy vấn trực tiếp số lượng dòng trong PostgreSQL để xác thực dữ liệu được import đầy đủ và khớp với kết quả phân tích dữ liệu ở Notebook 01.

In [7]:
# Danh sách các bảng thô và bảng tổng hợp cần đối chiếu số lượng dòng
tables_to_check = list(csv_mapping.keys()) + [
    'agg_installments_summary', 
    'agg_pos_cash_summary', 
    'agg_credit_card_summary'
]

validation_results = []
with conn.cursor() as cursor:
    for table in tables_to_check:
        try:
            cursor.execute(sql.SQL("SELECT COUNT(*) FROM {}").format(sql.Identifier(table)))
            row_count = cursor.fetchone()[0]
            validation_results.append({'Tên Bảng / View': table, 'Số dòng thực tế': row_count})
        except Exception as e:
            validation_results.append({'Tên Bảng / View': table, 'Số dòng thực tế': f"Lỗi: {e}"})

# Hiển thị kết quả kiểm chứng dưới dạng bảng Pandas
df_validation = pd.DataFrame(validation_results)
df_validation

,Tên Bảng / View,Số dòng thực tế
0,application_train,307511
1,application_test,48744
2,bureau,1716428
3,bureau_balance,27299925
4,previous_application,1670214
5,pos_cash_balance,10001358
6,installments_payments,13605401
7,credit_card_balance,3840312
8,agg_installments_summary,339587
9,agg_pos_cash_summary,337252


**Nhận xét:**

Dữ liệu đã được nạp đầy đủ vào database PostgreSQL. Cụ thể:
- Bảng trung tâm `application_train` chứa đúng **307,511 dòng** và `application_test` chứa đúng **48,744 dòng**.
- Số lượng dòng trong các bảng phụ (như `installments_payments` có hơn **13.6 triệu dòng**, `pos_cash_balance` có hơn **10 triệu dòng**) hoàn toàn khớp với kích thước bộ dữ liệu thô ban đầu.
- Các bảng tổng hợp Materialized Views (`agg_installments_summary`, `agg_pos_cash_summary`, `agg_credit_card_summary`) đã được tạo và đánh index đầy đủ tương ứng với số lượng khách hàng duy nhất có giao dịch.

## 6. Đóng kết nối Database

Chúng ta ngắt kết nối để giải phóng tài nguyên hệ thống.

In [8]:
if conn:
    conn.close()
    print("🔌 Đã đóng kết nối PostgreSQL thành công.")

🔌 Đã đóng kết nối PostgreSQL thành công.


## 7. Tổng kết

Trong notebook này, chúng ta đã:
1. **Thiết lập kết nối an toàn:** Dùng thư viện `python-dotenv` để ẩn thông tin kết nối nhạy cảm.
2. **Tạo bảng dữ liệu thô:** Chạy file `01_create_tables.sql` thành công từ Python.
3. **Xây dựng Pipeline Import nhanh chóng:** Sử dụng `copy_expert` nạp dữ liệu từ CSV local trực tiếp vào PostgreSQL thành công mà không gặp lỗi phân quyền (Permission Denied) trên Windows.
4. **Triển khai Schema mở rộng:** Khởi tạo thành công View phân tích, bảng tổng hợp Materialized View và thiết lập Indexes tăng hiệu năng JOIN phục vụ cho Feature Engineering.

**Bước tiếp theo:** Dữ liệu CSDL đã sẵn sàng, chúng ta sẽ bắt đầu thực hiện làm sạch dữ liệu tại notebook kế tiếp: **03. Data Cleaning**.